1. Import All Libraries

In [2]:
import pandas as pd #loads and manipulates the Excel dataset
import numpy as np #provides numerical operations and array handling
from sklearn.ensemble import RandomForestClassifier #imports Random Forest model
from sklearn.linear_model import LogisticRegression # imports Logistic Regression model
from sklearn.tree import DecisionTreeClassifier #imports Decision Tree model
from sklearn.model_selection import train_test_split, GridSearchCV #splits data into training and testing sets. GridSearchCV tries many combinations of hyperparameters to find the best ones
from sklearn.preprocessing import LabelEncoder #converts text categories like "Male"/"Female" into numbers 0 and 1
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    ConfusionMatrixDisplay #tools to measure how good the model is
)
from imblearn.over_sampling import SMOTE #handles class imbalance by creating synthetic samples of the minority class
from xgboost import XGBClassifier #provides XGBoost model which typically outperforms Random Forest on tabular data
import matplotlib #draws the comparison charts and confusion matrix
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import joblib #saves and loads the trained model as a .pkl file
import warnings
warnings.filterwarnings('ignore') #hides non-critical warning messages to keep output clean

2. Load and Understand the Data : Load the Dataset

In [3]:
"""
Step 1: Load dataset and subset to the 11 selected features (+ CustomerID, Churn)
Step 2: Clean — median-impute missing numerics, encode Gender & MaritalStatus
"""
import pandas as pd

# ---- Step 1: Load & subset ----
df_raw = pd.read_excel(
    "../datasets/E Commerce Dataset.xlsx",
    sheet_name="E Comm"
)

KEEP = [
    'CustomerID', 'Churn',
    'Tenure', 'Gender', 'HourSpendOnApp', 'SatisfactionScore',
    'MaritalStatus', 'NumberOfAddress', 'Complain', 'CouponUsed',
    'OrderCount', 'DaySinceLastOrder', 'CashbackAmount',
]

df = df_raw[KEEP].copy()

print('=== STEP 1: Subset shape ===')
print(df.shape)
print(df.dtypes)
print()

print('=== STEP 1: Missing values BEFORE cleaning ===')
print(df.isnull().sum())
print()

=== STEP 1: Subset shape ===
(5630, 13)
CustomerID             int64
Churn                  int64
Tenure               float64
Gender                object
HourSpendOnApp       float64
SatisfactionScore      int64
MaritalStatus         object
NumberOfAddress        int64
Complain               int64
CouponUsed           float64
OrderCount           float64
DaySinceLastOrder    float64
CashbackAmount       float64
dtype: object

=== STEP 1: Missing values BEFORE cleaning ===
CustomerID             0
Churn                  0
Tenure               264
Gender                 0
HourSpendOnApp       255
SatisfactionScore      0
MaritalStatus          0
NumberOfAddress        0
Complain               0
CouponUsed           256
OrderCount           258
DaySinceLastOrder    307
CashbackAmount         0
dtype: int64



3. Cleaning the dataset (median-impute, encode Gender/MaritalStatus)

In [4]:
# ---- Step 2: Clean ----

# 2a. Median-impute the 5 numeric features with missing values
impute_cols = ['Tenure', 'HourSpendOnApp', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder']
medians = {}
for col in impute_cols:
    med = df[col].median()
    medians[col] = med
    df[col] = df[col].fillna(med)

print('=== STEP 2a: Medians used for imputation ===')
for k, v in medians.items():
    print(f'  {k}: {v}')
print()

# 2b. Encode Gender (Male/Female -> 1/0) and MaritalStatus (Married/Single/Divorced -> 1/2/0)
df['Gender_encoded'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['MaritalStatus_encoded'] = df['MaritalStatus'].map({'Married': 1, 'Single': 2, 'Divorced': 0})

print('=== STEP 2b: Encoding check ===')
print(df[['Gender', 'Gender_encoded']].drop_duplicates())
print(df[['MaritalStatus', 'MaritalStatus_encoded']].drop_duplicates())
print()

print('=== STEP 2: Missing values AFTER cleaning ===')
print(df.isnull().sum())
print()

print('=== STEP 2: Duplicate check (full row, excluding CustomerID) ===')
print('Duplicates:', df.drop(columns=['CustomerID']).duplicated().sum())
print()

print('=== STEP 2: Final cleaned dataset preview ===')
print(df.head(10))

# Save cleaned dataset for Step 3 (EDA) and later model training
df.to_csv('../datasets/churn_11features_cleaned.csv', index=False)
print()
print('Saved -> churn_11features_cleaned.csv')
print('Final shape:', df.shape)

=== STEP 2a: Medians used for imputation ===
  Tenure: 9.0
  HourSpendOnApp: 3.0
  CouponUsed: 1.0
  OrderCount: 2.0
  DaySinceLastOrder: 3.0

=== STEP 2b: Encoding check ===
   Gender  Gender_encoded
0  Female               0
1    Male               1
   MaritalStatus  MaritalStatus_encoded
0         Single                      2
6       Divorced                      0
15       Married                      1

=== STEP 2: Missing values AFTER cleaning ===
CustomerID               0
Churn                    0
Tenure                   0
Gender                   0
HourSpendOnApp           0
SatisfactionScore        0
MaritalStatus            0
NumberOfAddress          0
Complain                 0
CouponUsed               0
OrderCount               0
DaySinceLastOrder        0
CashbackAmount           0
Gender_encoded           0
MaritalStatus_encoded    0
dtype: int64

=== STEP 2: Duplicate check (full row, excluding CustomerID) ===
Duplicates: 623

=== STEP 2: Final cleaned dataset previ

4. EDA — restricted to the 11 selected features + Churn

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

print("=" * 60)
print("STEP 3: EXPLORATORY DATA ANALYSIS (11 FEATURES ONLY)")
print("=" * 60)

df = pd.read_csv('../datasets/churn_11features_cleaned.csv')

numeric_features = ['Tenure', 'HourSpendOnApp', 'SatisfactionScore', 'NumberOfAddress',
                     'Complain', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']
categorical_features = ['Gender', 'MaritalStatus']

STEP 3: EXPLORATORY DATA ANALYSIS (11 FEATURES ONLY)


In [6]:
# ---------------------------------------------------------
# 3a. Churn distribution (total churn vs non-churn)
# ---------------------------------------------------------
churn_counts = df['Churn'].value_counts().sort_index()
churn_pct = df['Churn'].value_counts(normalize=True).sort_index() * 100

print('\n--- Churn distribution ---')
print(f"Non-Churn (0): {churn_counts[0]}  ({churn_pct[0]:.2f}%)")
print(f"Churn (1):     {churn_counts[1]}  ({churn_pct[1]:.2f}%)")

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(['Not Churn', 'Churn'], churn_counts.values, color=['#2563eb', '#dc2626'])
for bar, count, pct in zip(bars, churn_counts.values, churn_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{count}\n({pct:.1f}%)', ha='center', fontweight='bold')
ax.set_title('Churn vs Non-Churn — Total Customers', fontsize=14)
ax.set_ylabel('Number of Customers')
plt.tight_layout()
plt.savefig('../plots/eda_01_churn_distribution.png', dpi=120)
plt.show()


--- Churn distribution ---
Non-Churn (0): 4682  (83.16%)
Churn (1):     948  (16.84%)


In [7]:
# ---------------------------------------------------------
# 3b. Descriptive stats table for numeric features
# ---------------------------------------------------------
desc = df[numeric_features].agg(['mean', 'median', 'std', 'min', 'max'])
mode_row = df[numeric_features].mode().iloc[0]
desc.loc['mode'] = mode_row
desc = desc.round(2)
print('\n--- Descriptive stats (11-feature numeric subset) ---')
print(desc)
desc.to_csv('../datasets/eda_descriptive_stats.csv')



--- Descriptive stats (11-feature numeric subset) ---
        Tenure  HourSpendOnApp  SatisfactionScore  NumberOfAddress  Complain  \
mean     10.13            2.93               3.07             4.21      0.28   
median    9.00            3.00               3.00             3.00      0.00   
std       8.36            0.71               1.38             2.58      0.45   
min       0.00            0.00               1.00             1.00      0.00   
max      61.00            5.00               5.00            22.00      1.00   
mode      1.00            3.00               3.00             2.00      0.00   

        CouponUsed  OrderCount  DaySinceLastOrder  CashbackAmount  
mean          1.72        2.96               4.46          177.22  
median        1.00        2.00               3.00          163.28  
std           1.86        2.88               3.57           49.21  
min           0.00        1.00               0.00            0.00  
max          16.00       16.00              

In [8]:
# ---------------------------------------------------------
# 3c. Univariate distributions — numeric features
# ---------------------------------------------------------
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
for i, col in enumerate(numeric_features):
    sns.histplot(df[col], kde=True, ax=axes[i], color='#2563eb')
    axes[i].set_title(col)
plt.tight_layout()
plt.savefig('../plots/eda_02_univariate_distributions.png', dpi=120)
plt.show()

In [9]:
# ---------------------------------------------------------
# 3d. Categorical feature counts
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(data=df, x='Gender', ax=axes[0], palette=['#f472b6', '#60a5fa'])
axes[0].set_title('Gender distribution')
sns.countplot(data=df, x='MaritalStatus', ax=axes[1], palette='Blues_d')
axes[1].set_title('Marital Status distribution')
plt.tight_layout()
plt.savefig('../plots/eda_03_categorical_distributions.png', dpi=120)
plt.show()

In [10]:
# ---------------------------------------------------------
# 3e. Bivariate — mean of each numeric feature by churn
# ---------------------------------------------------------
bivar = df.groupby('Churn')[numeric_features].mean().round(2).T
bivar.columns = ['Not Churn (0)', 'Churn (1)']
print('\n--- Mean of each feature, split by Churn ---')
print(bivar)
bivar.to_csv('../datasets/eda_bivariate_means_by_churn.csv')

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
for i, col in enumerate(numeric_features):
    sns.boxplot(data=df, x='Churn', y=col, ax=axes[i], palette=['#2563eb', '#dc2626'])
    axes[i].set_xticklabels(['Not Churn', 'Churn'])
    axes[i].set_title(col)
plt.tight_layout()
plt.savefig('../plots/eda_04_boxplots_by_churn.png', dpi=120)
plt.show()


--- Mean of each feature, split by Churn ---
                   Not Churn (0)  Churn (1)
Tenure                     11.40       3.86
HourSpendOnApp              2.93       2.96
SatisfactionScore           3.00       3.39
NumberOfAddress             4.16       4.47
Complain                    0.23       0.54
CouponUsed                  1.72       1.71
OrderCount                  2.99       2.81
DaySinceLastOrder           4.71       3.22
CashbackAmount            180.64     160.37


In [11]:
# ---------------------------------------------------------
# 3f. Churn rate by categorical features
# ---------------------------------------------------------
gender_churn = df.groupby('Gender')['Churn'].mean().round(3) * 100
marital_churn = df.groupby('MaritalStatus')['Churn'].mean().round(3) * 100
print('\n--- Churn rate (%) by Gender ---')
print(gender_churn)
print('\n--- Churn rate (%) by Marital Status ---')
print(marital_churn)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
gender_churn.plot(kind='bar', ax=axes[0], color='#dc2626')
axes[0].set_title('Churn rate (%) by Gender')
axes[0].set_ylabel('Churn rate (%)')
marital_churn.plot(kind='bar', ax=axes[1], color='#dc2626')
axes[1].set_title('Churn rate (%) by Marital Status')
axes[1].set_ylabel('Churn rate (%)')
plt.tight_layout()
plt.savefig('../plots/eda_05_churn_rate_by_category.png', dpi=120)
plt.show()


--- Churn rate (%) by Gender ---
Gender
Female    15.5
Male      17.7
Name: Churn, dtype: float64

--- Churn rate (%) by Marital Status ---
MaritalStatus
Divorced    14.6
Married     11.5
Single      26.7
Name: Churn, dtype: float64


In [12]:
# ---------------------------------------------------------
# 3g. Correlation heatmap (numeric + encoded categoricals + Churn)
# ---------------------------------------------------------
corr_cols = numeric_features + ['Gender_encoded', 'MaritalStatus_encoded', 'Churn']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Heatmap — 11 Selected Features + Churn')
plt.tight_layout()
plt.savefig('../plots/eda_06_correlation_heatmap.png', dpi=120)
plt.show()

print('\n--- Correlation of each feature with Churn (sorted by strength) ---')
print(corr['Churn'].drop('Churn').sort_values(key=abs, ascending=False))

print('\nAll EDA charts saved to ../plots/, summary CSVs saved to ../datasets/')


--- Correlation of each feature with Churn (sorted by strength) ---
Tenure                  -0.337831
Complain                 0.250188
DaySinceLastOrder       -0.155871
CashbackAmount          -0.154118
MaritalStatus_encoded    0.140316
SatisfactionScore        0.105481
NumberOfAddress          0.043931
Gender_encoded           0.029264
OrderCount              -0.024038
HourSpendOnApp           0.018816
CouponUsed              -0.001430
Name: Churn, dtype: float64

All EDA charts saved to ../plots/, summary CSVs saved to ../datasets/


5. Feature Engineering - Create New Features From Existing Ones
What is feature engineering? Creating new columns by combining or transforming existing ones to give the model stronger, more direct signals. Raw columns on their own can be weak — OrderCount=2 alone doesn't tell the model much, but order_rate = 2 orders ÷ 24 months tenure = 0.08 immediately says "barely buys despite being a long-time member," which is a much sharper churn signal.

We start from churn_11features_cleaned.csv (our 11 selected, cleaned features) and add 8 new engineered columns on top, ending with 19 total features.

In [18]:
print("=" * 60)
print("STEP 4: FEATURE ENGINEERING")
print("=" * 60)

df = pd.read_csv('../datasets/churn_11features_cleaned.csv')

STEP 4: FEATURE ENGINEERING


i. recency_risk
(DaySinceLastOrder > 10).astype(int) → 1 if the customer hasn't ordered in more than 10 days, else 0. In e-commerce, 10+ days of silence after typically frequent ordering is an early churn warning sign. .astype(int) converts the True/False result into 1/0 so the model can use it numerically.

In [19]:
# ── Recency feature ───────────────────────────────
# 1 if customer hasn't ordered in >10 days — early warning sign
df['recency_risk'] = (df['DaySinceLastOrder'] > 10).astype(int)
print("  + recency_risk")

  + recency_risk


ii. order_rate
OrderCount / (Tenure + 1) → orders placed per month of membership. Distinguishes a customer with 2 orders in their first month (rate = 1.0, healthy) from one with 2 orders across 24 months (rate ≈ 0.08, at risk) — OrderCount alone can't tell these apart. The +1 prevents a divide-by-zero error for brand-new customers with Tenure = 0.

In [20]:
# ── Order frequency ───────────────────────────────
# Orders placed per month of membership. +1 avoids divide-by-zero.
df['order_rate'] = df['OrderCount'] / (df['Tenure'] + 1)
print("  + order_rate")

  + order_rate


iii. low_order_flag
(OrderCount <= 2).astype(int) → 1 if the customer has placed 2 or fewer orders total. Customers who barely buy are the easiest to lose, so this gives the model one clean binary flag instead of relying on it to infer the threshold itself.

In [21]:
# ── Low order flag ────────────────────────────────
# 1 if customer has placed 2 or fewer orders total — barely-buying signal
df['low_order_flag'] = (df['OrderCount'] <= 2).astype(int)
print("  + low_order_flag")

  + low_order_flag


iv. cashback_per_month
CashbackAmount / (Tenure + 1) → how much cashback reward the customer earns per month of membership. Low cashback-per-month can mean the customer isn't spending enough to benefit from the loyalty program, weakening their reason to stay.

In [22]:
# ── Monetary feature ──────────────────────────────
# Cashback earned per month of membership
df['cashback_per_month'] = df['CashbackAmount'] / (df['Tenure'] + 1)
print("  + cashback_per_month")

  + cashback_per_month


v. coupon_usage_rate
CouponUsed / (OrderCount + 1) → coupons used per order. A high ratio paired with few total orders flags a price-sensitive customer who may leave once the deals stop — different from someone who orders frequently and uses coupons.

In [23]:
# ── Coupon usage rate ─────────────────────────────
# Coupons used per order. High usage + low orders = price-sensitive churn pattern.
df['coupon_usage_rate'] = df['CouponUsed'] / (df['OrderCount'] + 1)
print("  + coupon_usage_rate")

  + coupon_usage_rate


vi. low_engagement
(HourSpendOnApp < 2).astype(int) → 1 if the customer spends less than 2 hours on the app. Low app time is a direct disengagement signal — they're not browsing, not interested, not being re-exposed to products.

In [24]:
# ── Engagement flag ───────────────────────────────
# 1 if customer spends less than 2 hours on the app — disengagement signal
df['low_engagement'] = (df['HourSpendOnApp'] < 2).astype(int)
print("  + low_engagement")

  + low_engagement


vii. engagement_score
A weighted blend: HourSpendOnApp × 0.4 + OrderCount × 0.4 + NumberOfAddress × 0.1 + CouponUsed × 0.1. This combines several weaker individual signals into one more stable composite score — OrderCount and HourSpendOnApp get the highest weights since buying and active app use are the strongest engagement indicators.

In [27]:
# ── Engagement score ──────────────────────────────
# NOTE: original notebook used NumberOfDeviceRegistered here, which we
# dropped from our 11 features. Replaced with NumberOfAddress as the
# supporting signal instead — still a behavioral/account-activity proxy.
df['engagement_score'] = (
    df['HourSpendOnApp']    * 0.4 +
    df['OrderCount']        * 0.4 +
    df['NumberOfAddress']   * 0.1 +
    df['CouponUsed']        * 0.1
)
print("  + engagement_score (using NumberOfAddress in place of NumberOfDeviceRegistered)")


  + engagement_score (using NumberOfAddress in place of NumberOfDeviceRegistered)


viii. dissatisfied
(SatisfactionScore <= 2) | (Complain == 1) → 1 if the customer rated their satisfaction 1 or 2, or filed a complaint (the | is a logical OR — either condition alone triggers it). Combines two related dissatisfaction signals into one strong flag, since either signal independently can mean an unhappy customer.

In [28]:
# ── Satisfaction risk ─────────────────────────────
# 1 if satisfaction score <= 2 OR customer complained
df['dissatisfied'] = (
    (df['SatisfactionScore'] <= 2) | (df['Complain'] == 1)
).astype(int)
print("  + dissatisfied")

  + dissatisfied


ix. churn_risk_score — the master composite
recency_risk×0.30 + low_order_flag×0.25 + low_engagement×0.25 + dissatisfied×0.20. A single weighted summary of all four engineered risk flags. The weights reflect relative importance: not ordering recently (0.30) is treated as the single strongest signal, rarely buying and low app engagement are tied as the next-strongest (0.25 each), and dissatisfaction/complaints matter but somewhat less (0.20) since some complainers keep buying anyway. This gives the model one clean, pre-digested risk indicator to use alongside the 18 individual features.

In [29]:
# ── Combined churn risk signal ────────────────────
# Master score combining the engineered binary flags
df['churn_risk_score'] = (
    df['recency_risk']    * 0.30 +
    df['low_order_flag']  * 0.25 +
    df['low_engagement']  * 0.25 +
    df['dissatisfied']    * 0.20
)
print("  + churn_risk_score")

print(f"\nTotal features now: {len(df.columns) - 2} (11 original + 8 engineered, minus CustomerID/Churn)")
print(f"Dataset shape: {df.shape}")

  + churn_risk_score

Total features now: 22 (11 original + 8 engineered, minus CustomerID/Churn)
Dataset shape: (5630, 24)


Final output: 19 total columns feeding the model (11 original cleaned features + 8 engineered), saved to churn_11features_engineered.csv, ready for the train/test split.

In [30]:
# Save for the split step
df.to_csv('../datasets/churn_11features_engineered.csv', index=False)
print("\nSaved -> ../datasets/churn_11features_engineered.csv")


Saved -> ../datasets/churn_11features_engineered.csv


6. Train/test split
- Purpose: hold out data the model never sees during training, so we can fairly test if it actually learned patterns vs. just memorized
- Loads the 19-feature engineered dataset from Step 4
- Builds X (features): drops CustomerID (not a real signal), Churn (that's the target, not an input), and raw Gender/MaritalStatus text columns (already have encoded numeric versions)
- Builds y (target): just the Churn column (1 = churned, 0 = not)
- Prints column list — confirms exactly which 19 features go into the model, in order
- Split: 80% train / 20% test
random_state=42 → reproducible split every time
stratify=y → keeps the same ~83.2%/16.8% churn ratio in both train and test sets (prevents an unlucky random split from skewing one set)
- Ratio check at the end verifies stratification worked correctly
- Must run after feature engineering, not before — so X_train and X_test both have all 19 columns; engineering features only on the train set would leave test missing columns and break evaluation

In [17]:
print("=" * 60)
print("STEP 5: TRAIN TEST SPLIT")
print("=" * 60)

from sklearn.model_selection import train_test_split

df = pd.read_csv('../datasets/churn_11features_engineered.csv')

X = df.drop(columns=['CustomerID', 'Churn', 'Gender', 'MaritalStatus'])
y = df['Churn']

print(f"\nFinal feature columns going into the model ({X.shape[1]} total):")
for i, col in enumerate(X.columns):
    print(f"  {i}: {col}")

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"\nTrain size: {X_train.shape[0]}   Test size: {X_test.shape[0]}")
print("\nChurn ratio check (should stay ~83.2% / 16.8% in both splits):")
print("Train:")
print((y_train.value_counts(normalize=True) * 100).round(2))
print("Test:")
print((y_test.value_counts(normalize=True) * 100).round(2))

STEP 5: TRAIN TEST SPLIT

Final feature columns going into the model (20 total):
  0: Tenure
  1: HourSpendOnApp
  2: SatisfactionScore
  3: NumberOfAddress
  4: Complain
  5: CouponUsed
  6: OrderCount
  7: DaySinceLastOrder
  8: CashbackAmount
  9: Gender_encoded
  10: MaritalStatus_encoded
  11: recency_risk
  12: order_rate
  13: low_order_flag
  14: cashback_per_month
  15: coupon_usage_rate
  16: low_engagement
  17: engagement_score
  18: dissatisfied
  19: churn_risk_score

X shape: (5630, 20)
y shape: (5630,)

Train size: 4504   Test size: 1126

Churn ratio check (should stay ~83.2% / 16.8% in both splits):
Train:
Churn
0    83.17
1    16.83
Name: proportion, dtype: float64
Test:
Churn
0    83.13
1    16.87
Name: proportion, dtype: float64


6. SMOTE — Fixing Class Imbalance
- Before SMOTE: training set was imbalanced — 3,746 Not Churn vs only 758 Churn (~83%/17%)
- After SMOTE: perfectly balanced — 3,746 vs 3,746, by generating synthetic churn examples interpolated between real churn customers' feature values
- Test set untouched — X_test/y_test still reflect the real-world ~83%/17% ratio, so evaluation later stays honest
- Train shape grew from (4504, 20) → (7492, 20) — row count nearly doubled, column count unchanged (SMOTE only adds rows, never columns)

In [31]:
print("=" * 60)
print("STEP 6: SMOTE — FIXING CLASS IMBALANCE")
print("=" * 60)

from imblearn.over_sampling import SMOTE

print(f"\nBefore SMOTE:")
print(f"  Not Churn (0): {(y_train == 0).sum()}")
print(f"  Churn     (1): {(y_train == 1).sum()}")

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE:")
print(f"  Not Churn (0): {(y_train_bal == 0).sum()}")
print(f"  Churn     (1): {(y_train_bal == 1).sum()}")
print(f"\nTrain shape before: {X_train.shape}  ->  after: {X_train_bal.shape}")

STEP 6: SMOTE — FIXING CLASS IMBALANCE

Before SMOTE:
  Not Churn (0): 3746
  Churn     (1): 758

After SMOTE:
  Not Churn (0): 3746
  Churn     (1): 3746

Train shape before: (4504, 20)  ->  after: (7492, 20)


In [32]:
X_train.columns.tolist()

['Tenure',
 'HourSpendOnApp',
 'SatisfactionScore',
 'NumberOfAddress',
 'Complain',
 'CouponUsed',
 'OrderCount',
 'DaySinceLastOrder',
 'CashbackAmount',
 'Gender_encoded',
 'MaritalStatus_encoded',
 'recency_risk',
 'order_rate',
 'low_order_flag',
 'cashback_per_month',
 'coupon_usage_rate',
 'low_engagement',
 'engagement_score',
 'dissatisfied',
 'churn_risk_score']

7. Train and Compare 5 Algorithms (on 20 features)
What this step does: trains all 4 algorithms on the SMOTE-balanced training set (20 features now, including the 9 engineered ones), then evaluates each on the untouched real-world test set using 5 metrics — giving a fair side-by-side comparison to pick the best base model before tuning.

XGBoost wins overall — best accuracy, AUC-ROC, F1, and precision. Still the right choice for tuning in Step 8.

In [33]:
print("=" * 60)
print("STEP 7: TRAINING AND COMPARING MODELS")
print("=" * 60)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, recall_score, f1_score, precision_score,
    confusion_matrix, classification_report,
)

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    'Decision Tree': DecisionTreeClassifier(
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    'XGBoost': XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric='logloss'
    ),
}

results = {}

print(f"\n{'Model':<25} {'Accuracy':>10} {'AUC-ROC':>10} "
      f"{'Recall':>10} {'F1':>10} {'Precision':>12}")
print("-" * 75)

for name, model in models.items():
    model.fit(X_train_bal, y_train_bal)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)

    results[name] = {
        'model': model, 'accuracy': acc, 'auc': auc,
        'recall': rec, 'f1': f1, 'precision': prec,
    }

    print(f"{name:<25} {acc:>10.4f} {auc:>10.4f} {rec:>10.4f} {f1:>10.4f} {prec:>12.4f}")

print("\n--- Why we care most about Recall for the churn class (1) ---")
print("Recall = of all customers who actually churned, how many did we catch.")
print("Missing a churner (false negative) = a lost customer we never tried to save.")
print("A false alarm (false positive) = an unnecessary retention email, low cost.")
print("So a model with high recall on churn, even at the cost of some precision, is preferred.")

STEP 7: TRAINING AND COMPARING MODELS

Model                       Accuracy    AUC-ROC     Recall         F1    Precision
---------------------------------------------------------------------------
Logistic Regression           0.8188     0.8496     0.7263     0.5750       0.4759
Decision Tree                 0.9432     0.9155     0.8737     0.8384       0.8058
Random Forest                 0.9494     0.9812     0.8368     0.8480       0.8595
XGBoost                       0.9565     0.9836     0.8368     0.8665       0.8983

--- Why we care most about Recall for the churn class (1) ---
Recall = of all customers who actually churned, how many did we catch.
Missing a churner (false negative) = a lost customer we never tried to save.
A false alarm (false positive) = an unnecessary retention email, low cost.
So a model with high recall on churn, even at the cost of some precision, is preferred.


8. Hyperparameter Tuning — GridSearchCV on XGBoost (20 features)
- What are hyperparameters: Settings that control how the model learns. They are not learned from data — we set them before training. The default values are often not the best ones.
- What GridSearchCV does: Tries every possible combination of the parameters we specify, trains and tests each one using cross-validation, and tells us which combination performed best.

- Goal: find the best combination of XGBoost's settings (hyperparameters) rather than relying on defaults, to push performance beyond Step 7's untuned XGBoost (AUC 0.9836).
- param_grid: 108 total combinations of n_estimators (100/200/300 trees), max_depth (3/5/7, how complex each tree can get), learning_rate (0.01/0.05/0.1, how much each tree corrects the last), subsample (0.8/1.0, fraction of rows per tree), colsample_bytree (0.8/1.0, fraction of columns per tree).
- GridSearchCV tries every combination using 5-fold cross-validation (108 × 5 = 540 model fits), scoring by roc_auc, and keeps the best one. n_jobs=-1 parallelizes across all CPU cores to speed this up.
- grid_search.best_estimator_ is the already-trained winning model — no need to refit.
- Final evaluation runs that tuned model on the real (non-SMOTE) test set and reports the full picture: accuracy, AUC-ROC, recall, F1, precision, classification report, and a confusion matrix breakdown (TN/FP/FN/TP) so you can see exactly how many churners were caught vs missed.


In [34]:
print("=" * 60)
print("STEP 8: HYPERPARAMETER TUNING — XGBOOST")
print("=" * 60)

from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators':     [100, 200, 300],
    'max_depth':        [3, 5, 7],
    'learning_rate':    [0.01, 0.05, 0.1],
    'subsample':        [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}

print("\nParameters being tested:")
total_combinations = 1
for param, values in param_grid.items():
    print(f"  {param}: {values}")
    total_combinations *= len(values)
print(f"\nTotal combinations: {total_combinations}")
print(f"With 5-fold CV: {total_combinations * 5} model fits")
print("This will take a few minutes...\n")

xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train_bal, y_train_bal)

print(f"\nBest parameters found:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nBest cross-validation AUC: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_

# Evaluate the tuned model on the untouched real-world test set
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

final_acc = accuracy_score(y_test, y_pred)
final_auc = roc_auc_score(y_test, y_proba)
final_recall = recall_score(y_test, y_pred)
final_f1 = f1_score(y_test, y_pred)
final_precision = precision_score(y_test, y_pred)

print("\n" + "=" * 60)
print("STEP 9: FINAL MODEL EVALUATION — XGBOOST TUNED")
print("=" * 60)
print(f"Accuracy:  {final_acc:.4f}")
print(f"AUC-ROC:   {final_auc:.4f}")
print(f"Recall:    {final_recall:.4f}")
print(f"F1:        {final_f1:.4f}")
print(f"Precision: {final_precision:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Churn', 'Churn']))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(f"  True Negative  (correctly predicted Not Churn): {cm[0][0]}")
print(f"  False Positive (predicted Churn, was Not Churn): {cm[0][1]}")
print(f"  False Negative (predicted Not Churn, was Churn): {cm[1][0]}")
print(f"  True Positive  (correctly predicted Churn):      {cm[1][1]}")

STEP 8: HYPERPARAMETER TUNING — XGBOOST

Parameters being tested:
  n_estimators: [100, 200, 300]
  max_depth: [3, 5, 7]
  learning_rate: [0.01, 0.05, 0.1]
  subsample: [0.8, 1.0]
  colsample_bytree: [0.8, 1.0]

Total combinations: 108
With 5-fold CV: 540 model fits
This will take a few minutes...

Fitting 5 folds for each of 108 candidates, totalling 540 fits

Best parameters found:
  colsample_bytree: 0.8
  learning_rate: 0.1
  max_depth: 7
  n_estimators: 300
  subsample: 1.0

Best cross-validation AUC: 0.9927

STEP 9: FINAL MODEL EVALUATION — XGBOOST TUNED
Accuracy:  0.9627
AUC-ROC:   0.9905
Recall:    0.8632
F1:        0.8865
Precision: 0.9111

Classification Report:
              precision    recall  f1-score   support

   Not Churn       0.97      0.98      0.98       936
       Churn       0.91      0.86      0.89       190

    accuracy                           0.96      1126
   macro avg       0.94      0.92      0.93      1126
weighted avg       0.96      0.96      0.96    

- Final Tuned Model
Best hyperparameters found: n_estimators=300, max_depth=7, learning_rate=0.1, subsample=1.0, colsample_bytree=0.8 — deeper trees (7 vs default 6), more of them (300), full use of training rows each tree, slight feature subsampling per tree to reduce overfitting.
Before vs after tuning (XGBoost, 20 features):
Metric Untuned (Step 7) Tuned (Step 8) Change
Accuracy  0.9565         0.9627       ↑
AUC-ROC.  0.9836         0.9905        ↑
Recall.   0.8368         0.8632        ↑
F1.       0.8665         0.8865        ↑
Precision 0.8983         0.9111        ↑
Tuning improved every single metric — this is the result you want to keep.


Reading the confusion matrix (1,126 test customers):

920 True Negatives — correctly identified as staying
164 True Positives — correctly caught as churners
26 False Negatives — churners the model missed (these are the costly ones — lost customers with no retention attempt)
16 False Positives — flagged as at-risk but actually stayed (low cost — just an unnecessary retention email)

Classification report read: of 190 actual churners, the model catches 164 (86%, recall) and when it predicts churn, it's right 91% of the time (precision). The 936 non-churners are handled even better (98% recall, 97% precision) — expected, since that's the majority class.

9. Visualization Plots

In [36]:
print("=" * 60)
print("STEP 9: VISUALIZATIONS")
print("=" * 60)

STEP 9: VISUALIZATIONS


What each plot does:
- Confusion matrix heatmap — visualizes the same 920/16/26/164 breakdown from Step 8, but as a color-coded grid instead of raw numbers, making it easy to see at a glance how dominant the correct predictions (diagonal) are vs. errors (off-diagonal).

In [37]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay

sns.set_style('whitegrid')

# ---------------------------------------------------------
# 9a. Confusion Matrix — Tuned XGBoost
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Not Churn', 'Churn']
)
disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
ax.set_title('Confusion Matrix — XGBoost Tuned (20 Features)', fontsize=14)
plt.tight_layout()
plt.savefig('../plots/confusion_matrix_tuned.png', dpi=120)
plt.show()
print("Saved -> ../plots/confusion_matrix_tuned.png")

Saved -> ../plots/confusion_matrix_tuned.png


- Model comparison bar chart — plots AUC-ROC for all 5 models side by side (Logistic Regression, Decision Tree, Random Forest, XGBoost untuned, XGBoost Tuned), so you can visually confirm tuning pushed XGBoost to the top.

In [38]:
# ---------------------------------------------------------
# 9b. Model Comparison — AUC-ROC across all models + tuned XGBoost
# ---------------------------------------------------------
comparison = {name: r['auc'] for name, r in results.items()}
comparison['XGBoost Tuned'] = final_auc

names = list(comparison.keys())
scores = list(comparison.values())
colors = ['#94a3b8', '#64748b', '#475569', '#2563eb', '#1e3a8a']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(names, scores, color=colors[:len(names)])
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{score:.4f}', ha='center', fontweight='bold')
ax.set_ylim(min(scores) - 0.05, 1.02)
ax.set_ylabel('AUC-ROC Score')
ax.set_title('Model Comparison — AUC-ROC Score (20 Features)', fontsize=14)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('../plots/model_comparison_20features.png', dpi=120)
plt.show()
print("Saved -> ../plots/model_comparison_20features.png")

Saved -> ../plots/model_comparison_20features.png


- Feature importance chart — shows which of the 20 features the tuned model relies on most. This is the most interesting one for your report: it'll tell you whether your engineered features (churn_risk_score, recency_risk, order_rate, etc.) actually earned their place near the top, validating the feature engineering step, or whether the original raw features (Tenure, DaySinceLastOrder, Complain) still dominate.

In [39]:
# ---------------------------------------------------------
# 9c. Feature Importance — Tuned XGBoost, top 12
# ---------------------------------------------------------
importances = best_model.feature_importances_
feature_names = X_train.columns

imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False).head(12)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(imp_df['feature'][::-1], imp_df['importance'][::-1], color='#2563eb')
ax.set_xlabel('Importance Score')
ax.set_title('Top 12 Feature Importances — XGBoost Tuned (20 Features)', fontsize=14)
plt.tight_layout()
plt.savefig('../plots/feature_importance_20features.png', dpi=120)
plt.show()
print("Saved -> ../plots/feature_importance_20features.png")

print("\n--- Top 12 features ranked ---")
print(imp_df.to_string(index=False))

Saved -> ../plots/feature_importance_20features.png

--- Top 12 features ranked ---
           feature  importance
    low_order_flag    0.144691
  churn_risk_score    0.114022
    HourSpendOnApp    0.089081
cashback_per_month    0.088818
            Tenure    0.083546
        CouponUsed    0.059272
        order_rate    0.054670
      dissatisfied    0.047784
 DaySinceLastOrder    0.043807
          Complain    0.037946
 SatisfactionScore    0.036700
    CashbackAmount    0.031293


Visualizations — Summary
- Confusion matrix: 920 correctly identified as staying, 164 correctly caught as churners, only 26 missed (false negatives) and 16 false alarms (false positives) out of 1,126 test customers — strongly diagonal, confirming the 96.3% accuracy / 99.05% AUC numbers visually.
- Model comparison chart: clean ascending staircase — Logistic Regression (0.8496) → Decision Tree (0.9155) → Random Forest (0.9812) → XGBoost untuned (0.9836) → XGBoost Tuned (0.9905). Confirms tuning gave a real, visible lift over the already-strong untuned XGBoost.
- Feature importance — this is the interesting one. Top 12, ranked:

low_order_flag
churn_risk_score
HourSpendOnApp
cashback_per_month
Tenure
CouponUsed
order_rate
dissatisfied
DaySinceLastOrder
Complain
SatisfactionScore
CashbackAmount

Feature importance — key takeaways
- churn_risk_score (#2) added real signal rather than cannibalizing — its source features (low_order_flag #1, dissatisfied #8) stayed prominent too.
- 6 of the top 12 are engineered features — validates Step 4 was worth doing.
- recency_risk and low_engagement dropped out of the top 12, likely absorbed into churn_risk_score.
- Gender_encoded, MaritalStatus_encoded, NumberOfAddress stayed weak — matches the EDA findings.

"Cannibalizing" here means one feature stealing the model's attention away from related features, making them seem useless even though they actually carry real information.
In our case: churn_risk_score is built from recency_risk, low_order_flag, low_engagement, and dissatisfied (it's literally a weighted sum of them). The worry was — since churn_risk_score already contains all that information bundled together, would the model just use the composite score and ignore its raw ingredients entirely? If that happened, recency_risk, low_order_flag, etc. would all show near-zero importance, because the model "eats" their signal through the composite instead of using them directly.
That's what cannibalization would look like: the parts losing importance because the whole replaced them.
What we found instead: low_order_flag is still #1, dissatisfied is still #8 — they kept their own value alongside churn_risk_score at #2. So no cannibalization happened; the composite feature added something extra on top, rather than just repackaging information the model already had.

10. Saving the final model artifacts

What this saves, and why each file matters:

- churn_model.pkl — the actual tuned XGBoost model object (best_model from Step 8), so it can be loaded and reused without retraining.
- feature_columns.pkl — the exact list and order of the 20 column names the model expects (Tenure, HourSpendOnApp, ..., churn_risk_score). This is critical: when Django builds a feature row for a live customer, it must produce columns in this exact order, or the model will silently misinterpret which number means what.
- encoding_maps.pkl — much smaller now than your old 18-feature version. Old version needed mappings for PreferredLoginDevice, PreferredPaymentMode, PreferedOrderCat, Gender, MaritalStatus (5 categorical columns). Now only Gender and MaritalStatus remain, since those were the only 2 categoricals in your kept 11 features.

In [40]:
print("=" * 60)
print("STEP 10: SAVING FINAL MODEL ARTIFACTS")
print("=" * 60)

import joblib
import os

model_dir = '../ml_models'
os.makedirs(model_dir, exist_ok=True)

# ---------------------------------------------------------
# 10a. Save the trained model
# ---------------------------------------------------------
model_path = os.path.join(model_dir, 'churn_model.pkl')
joblib.dump(best_model, model_path)
print(f"Saved model -> {model_path}")

# ---------------------------------------------------------
# 10b. Save the exact feature column order the model expects
# ---------------------------------------------------------
feature_columns = X_train.columns.tolist()
columns_path = os.path.join(model_dir, 'feature_columns.pkl')
joblib.dump(feature_columns, columns_path)
print(f"Saved feature columns -> {columns_path}")
print(f"  ({len(feature_columns)} columns): {feature_columns}")

# ---------------------------------------------------------
# 10c. Save encoding maps — only Gender & MaritalStatus now,
# since PreferredLoginDevice/PreferredPaymentMode/PreferedOrderCat
# were dropped from our 11 selected features.
# ---------------------------------------------------------
encoding_maps = {
    'Gender':        {'Male': 1, 'Female': 0},
    'MaritalStatus': {'Married': 1, 'Single': 2, 'Divorced': 0},
}
encoding_path = os.path.join(model_dir, 'encoding_maps.pkl')
joblib.dump(encoding_maps, encoding_path)
print(f"Saved encoding maps -> {encoding_path}")
print(f"  {encoding_maps}")

print("\n--- All artifacts saved successfully ---")
print(f"Model dir: {os.path.abspath(model_dir)}")
print("Files:")
for f in os.listdir(model_dir):
    print(f"  {f}")

STEP 10: SAVING FINAL MODEL ARTIFACTS
Saved model -> ../ml_models/churn_model.pkl
Saved feature columns -> ../ml_models/feature_columns.pkl
  (20 columns): ['Tenure', 'HourSpendOnApp', 'SatisfactionScore', 'NumberOfAddress', 'Complain', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount', 'Gender_encoded', 'MaritalStatus_encoded', 'recency_risk', 'order_rate', 'low_order_flag', 'cashback_per_month', 'coupon_usage_rate', 'low_engagement', 'engagement_score', 'dissatisfied', 'churn_risk_score']
Saved encoding maps -> ../ml_models/encoding_maps.pkl
  {'Gender': {'Male': 1, 'Female': 0}, 'MaritalStatus': {'Married': 1, 'Single': 2, 'Divorced': 0}}

--- All artifacts saved successfully ---
Model dir: /Users/a97798/Documents/e-commerce_churn_system/ecommerce/apps/churn/ml_models
Files:
  encoding_maps.pkl
  feature_columns.pkl
  __init__.py
  churn_model.pkl
